# Fire Detection Training in Google Colab

This notebook installs the required packages, downloads a public fire-and-smoke dataset from Roboflow in YOLOv8 format, trains a `YOLOv8s` model on GPU, prints validation metrics, and copies the best weights to Google Drive.


## 1. Install Dependencies

This section installs the `ultralytics` and `roboflow` packages needed for dataset download, training, and evaluation.


In [5]:
%pip install -q ultralytics roboflow


## 2. Download the Roboflow Dataset

This section connects to Roboflow using your API key and downloads the public Middle East Tech University fire-and-smoke dataset (version `2`) in YOLOv8 format. This dataset contains `15,345` labeled images, which comfortably exceeds the 1,000-image requirement.


In [6]:
from pathlib import Path

from roboflow import Roboflow

ROBOFLOW_API_KEY = "rWXcniwmqcsBvee2q7yP"
WORKSPACE = "middle-east-tech-university"
PROJECT = "fire-and-smoke-detection-hiwia"
VERSION = 2

rf = Roboflow(api_key="rWXcniwmqcsBvee2q7yP")
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")

data_yaml = Path(dataset.location) / "data.yaml"
print(f"Dataset downloaded to: {dataset.location}")
print(f"Using data config: {data_yaml}")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to fire-and-smoke-detection-2 in yolov8:: 100%|██████████| 30686/30686 [00:18<00:00, 1698.11it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Dataset downloaded to: /content/fire-and-smoke-detection-2
Using data config: /content/fire-and-smoke-detection-2/data.yaml


## 3. Train YOLOv8s on GPU

This section verifies that a GPU runtime is available in Colab and trains `yolov8s.pt` for `50` epochs at `640` image size on `device=0`.


In [7]:
import torch
from ultralytics import YOLO

assert torch.cuda.is_available(), "GPU not detected. In Colab, select Runtime -> Change runtime type -> GPU before running training."

model = YOLO("yolov8s.pt")
_ = model.train(
    data=str(data_yaml),
    epochs=50,
    imgsz=640,
    device=0,
    project="runs/detect",
    name="train",
    exist_ok=True,
)

print("Training completed. Best weights should be saved to /content/runs/detect/train/weights/best.pt")


Ultralytics 8.4.27 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/fire-and-smoke-detection-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100,

## 4. Validate the Best Checkpoint

This section loads the trained `best.pt` checkpoint, runs validation on the dataset, and prints `mAP50`, `precision`, and `recall`.


In [10]:
from pathlib import Path

from ultralytics import YOLO

best_model_path = Path("/content/runs/detect/runs/detect/train/weights/best.pt")
assert best_model_path.exists(), "best.pt was not found. Make sure training finished successfully before running validation."
best_model = YOLO(str(best_model_path))
metrics = best_model.val(data=str(data_yaml), imgsz=640, device=0, split="val")

print(f"mAP50: {metrics.box.map50:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")


Ultralytics 8.4.27 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,126,358 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2242.4±983.8 MB/s, size: 155.2 KB)
val: Scanning /content/fire-and-smoke-detection-2/valid/labels.cache... 1277 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1277/1277 281.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 80/80 4.7it/s 17.0s
                   all       1277       3931      0.629      0.585      0.597      0.251
                  fire        944       2121      0.649      0.656      0.661      0.267
                 smoke        980       1810      0.608      0.514      0.534      0.234
Speed: 1.0ms preprocess, 5.1ms inference, 0.0ms loss, 1.7ms postprocess per image
Results saved to /content/runs/detect/val
mAP50: 0.5972
Precision: 0.6287
Recall: 0.5848


## 5. Mount Google Drive and Save the Best Weights

This section mounts Google Drive, creates the destination folder if needed, and copies `runs/detect/train/weights/best.pt` to `/content/drive/MyDrive/fire_detection/best.pt`.


In [11]:
from pathlib import Path

from google.colab import drive
import shutil

drive.mount("/content/drive")

source_path = Path("/content/runs/detect/runs/detect/train/weights/best.pt")
assert source_path.exists(), "best.pt was not found. Make sure training finished successfully before copying the weights."
destination_dir = Path("/content/drive/MyDrive/fire_detection")
destination_dir.mkdir(parents=True, exist_ok=True)
destination_path = destination_dir / "best.pt"

shutil.copy2(source_path, destination_path)
print(f"Copied best weights to: {destination_path}")


Mounted at /content/drive


AssertionError: best.pt was not found. Make sure training finished successfully before copying the weights.